### Stored Procedure

In [2]:
USE Pampero
GO
CREATE OR ALTER PROC spEmpleados
AS
SELECT * FROM Empleados;

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.027

In [3]:
EXEC spEmpleados;

(9 rows affected)

IDEmpleado | Apellido  | Nombre   | Puesto                         | Saludo | FechaNacimiento | FechaAlta  | Direccion                     | Ciudad   | Region | CodigoPostal | Pais        | TelefonoCasa   | Interno | Notas                                                                                                                                                                                                    | JefeID | RutaFoto                              
-----------+-----------+----------+--------------------------------+--------+-----------------+------------+-------------------------------+----------+--------+--------------+-------------+----------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+---------------------------------------
1          | Davolio   | Nancy    | Represent

### Stored Procedure con parámetros

In [6]:
USE Pampero;
GO
CREATE OR ALTER PROC spEmpleadoPorNombre
@Apellido nvarchar(50)
AS
SELECT e.apellido, e.nombre, e.puesto, e.fechaalta
FROM Empleados e
WHERE e.apellido LIKE @Apellido + '%';


Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.039

In [7]:
EXEC spEmpleadoPorNombre 'King';

(1 row affected)

apellido | nombre | puesto                  | fechaalta 
---------+--------+-------------------------+-----------
King     | Robert | Representante de Ventas | 2017-01-02
(1 row)

Total execution time: 00:00:01.002

In [8]:
EXEC spEmpleadoPorNombre;

Msg 201, Level 16, State 4, Procedure spEmpleadoPorNombre, Line 0
Procedure or function 'spEmpleadoPorNombre' expects parameter '@Apellido', which was not supplied.

Total execution time: 00:00:00.004

#### Stored Procedure con parámetros por default

In [9]:
USE Pampero;
DROP PROC spEmpleadoPorNombre;
GO
CREATE PROC spEmpleadoPorNombre
@Apellido nvarchar(50) = NULL
AS
IF @Apellido IS NOT NULL
SELECT e.apellido, e.nombre, e.puesto, e.fechaalta
FROM Empleados e
WHERE e.apellido LIKE @Apellido + '%';
ELSE
SELECT e.apellido, e.nombre, e.puesto, e.fechaalta
FROM Empleados e

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.033

In [10]:
EXEC spEmpleadoPorNombre 'King';
EXEC spEmpleadoPorNombre;

(1 row affected)
(9 rows affected)

apellido | nombre | puesto                  | fechaalta 
---------+--------+-------------------------+-----------
King     | Robert | Representante de Ventas | 2017-01-02
(1 row)

apellido  | nombre   | puesto                         | fechaalta 
----------+----------+--------------------------------+-----------
Davolio   | Nancy    | Representante de Ventas        | 2015-05-01
Fuller    | Andrew   | Vicepresidente, Ventas         | 2015-08-14
Leverling | Janet    | Representante de Ventas        | 2015-04-01
Peacock   | Margaret | Representante de Ventas        | 2016-05-03
Buchanan  | Steven   | Gerente de Ventas              | 2016-10-17
Suyama    | Michael  | Representante de Ventas        | 2016-10-17
King      | Robert   | Representante de Ventas        | 2017-01-02
Callahan  | Laura    | Coordinador de Ventas Internas | 2017-03-05
Dodsworth | Anne     | Representante de Ventas        | 2017-11-15
(9 rows)

Total execution time: 00:00:00.016

In [11]:
CREATE PROC spTestReturns
AS
DECLARE @MiMensaje varchar(50);
DECLARE @MiOtroMensaje varchar(50);
SELECT @MiMensaje = 'Hola, esta es la linea antes del RETURN';
PRINT @MiMensaje;
RETURN;
SELECT @MiOtroMensaje = 'Perdon, pero esta linea no se ejecutara';
PRINT @MiOtroMensaje;
RETURN;

Commands completed successfully.

Total execution time: 00:00:00.693

In [12]:
EXEC spTestReturns

Hola, esta es la linea antes del RETURN

Total execution time: 00:00:00.003

## Triggers

In [13]:
CREATE TRIGGER productoordennodiscontinuado
ON [Detalles Pedido]
FOR INSERT, UPDATE
AS
IF EXISTS
(
SELECT 'Verdadero'
FROM Inserted i
JOIN Productos p
ON i.IDProducto = p.IDProducto
WHERE p.Discontinuado = 1
)
BEGIN
RAISERROR('El item de la orden ha sido discontinuado. La transaccion ha fallado.',16,1)
ROLLBACK TRAN
END

Commands completed successfully.

Total execution time: 00:00:00.017

In [14]:
INSERT INTO [Detalles Pedido] (IDPedido, IDProducto, Cantidad, PrecioUnitario)
VALUES (10248, 5, 10, 14.00);

Msg 50000, Level 16, State 1, Procedure productoordennodiscontinuado, Line 14
El item de la orden ha sido discontinuado. La transaccion ha fallado.
Msg 3609, Level 16, State 1, Line 1
The transaction ended in the trigger. The batch has been aborted.

Total execution time: 00:00:00.290

## Manejo de Errores

In [16]:
BEGIN TRY
    CREATE TABLE Prueba1(Col1 int PRIMARY KEY );
END TRY
BEGIN CATCH
    DECLARE @ErrorNo int,
    @Severity tinyint,
    @State smallint,
    @LineNo int,
    @Message nvarchar(4000);
    SELECT
    @ErrorNo = ERROR_NUMBER(), @Severity = ERROR_SEVERITY(), @State = ERROR_STATE(), @LineNo = ERROR_LINE (), @Message = ERROR_MESSAGE();
    IF @ErrorNo = 2714 -- El objeto existe
        PRINT 'WARNING: Saltando CREATE porque la tabla existe';
    ELSE 
        RAISERROR(@Message, 16, 1 );
END CATCH

Total execution time: 00:00:00.008

## Función: Obtener cantidad disponible de un producto

In [ ]:
CREATE OR ALTER FUNCTION obtener_cantidad_disponible(@producto_id INT) RETURNS INT
AS
BEGIN
    DECLARE @cantidad_disponible INT;
    SET @cantidad_disponible = (SELECT UnidadesEnStock - UnidadesEnPedidos AS cantidad_disponible
                                FROM Productos
                                WHERE IDProducto = @producto_id);
    RETURN @cantidad_disponible;
END;

Commands completed successfully.

Total execution time: 00:00:03.184

## Procedimiento Almacenado: Registrar un nuevo pedido

In [ ]:
CREATE OR ALTER PROCEDURE registrar_pedido
@clienteid INT,
@empid INT,
@fechaorden DATETIME,
@fecharequerida DATETIME,
@producto_id INT,
@preciounitario MONEY,
@cantidad INT,
@descuento REAL
AS
BEGIN
    DECLARE @cantidad_disponible INT;
    SET @cantidad_disponible = dbo.obtener_cantidad_disponible(@producto_id);

    IF @cantidad <= @cantidad_disponible
    BEGIN
    -- Comienzo una nueva transaccion
    BEGIN TRAN;
         -- Declaro una variable
        DECLARE @nuevopedidoid AS INT;
        -- Inserto un nuevo pedido en la tabla Pedidos
        INSERT INTO Pedidos
        (IDCliente, IDEmpleado, FechaPedido, FechaRequerida)
        VALUES
        (@clienteid, @empid, @fechaorden, @fecharequerida);
        -- Guardo el nuevo ID de pedido en una variable
        SET @nuevopedidoid = SCOPE_IDENTITY();
        -- Inserto lineas de pedido para el nuevo pedido en Detalles Pedido
        INSERT INTO [Detalles Pedido]
        (IDPedido, IDProducto, PrecioUnitario, Cantidad, Descuento)
        VALUES(@nuevopedidoid, @producto_id, @preciounitario, @cantidad, @descuento);
        -- Confirmo la transaccion
    COMMIT TRAN
    END
    ELSE
    BEGIN
        RAISERROR('No hay suficiente cantidad disponible', 16, 1);
    END;
END;

Commands completed successfully.

Total execution time: 00:00:00.039

## Trigger: Actualizar existencias al recibir un nuevo pedido

In [20]:
CREATE OR ALTER TRIGGER actualizar_existencias
ON [Detalles Pedido]
AFTER INSERT
AS
BEGIN
    DECLARE @cantidad INT;
    SELECT @cantidad = Cantidad FROM inserted;

    -- Actualizar las existencias en la tabla "productos"
    UPDATE Productos
    SET UnidadesEnPedidos = UnidadesEnPedidos + @cantidad
    WHERE IDProducto = (SELECT IDProducto FROM inserted);
END;

Commands completed successfully.

Total execution time: 00:00:00.020